# Channel audit: M4 screening of one model on a Colab or Kaggle GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arshia-HZ/mas-channel-audit/blob/main/notebooks/screen.ipynb)

This notebook screens **one model** on the frozen **screen split** (300 MATH test problems,
all levels): 4 samples per problem with the round-0 prompt, temperature 0.7 and the model
config's `max_tokens`. It reports accuracy by level, disagreement among the samples,
truncation and extraction failure, and gate G0 for the sanity model.

Run it **once per model**, changing `MODEL_CONFIG` in the settings:

| `MODEL_CONFIG` | Why |
| --- | --- |
| `configs/models/qwen3-1.7b.yaml` | primary candidate |
| `configs/models/qwen2.5-math-1.5b.yaml` | primary candidate |
| `configs/models/llama-3.2-3b-instruct.yaml` | second family (gated: needs an `HF_TOKEN` secret) |
| `configs/models/qwen2.5-1.5b-instruct.yaml` | sanity model for gate G0 only; set `SCREEN_PROMPTS = ["plain"]`; then again with `TEMPERATURE = 0` for the greedy G0 check |

Every step is **resumable**: after a disconnect, run the cells again. The problem split comes
from the pinned MATH test revision and is checked against the committed manifest, so all four
runs use the identical 300 problems. Nothing here chooses the primary model: that is your
decision, made from the comparison table and written into `PREREG.md`.

## 0. Settings

In [ ]:
MODEL_CONFIG = (
    "configs/models/qwen3-1.7b.yaml"  # one model per run, see the table above
)
BACKEND = (
    "vllm"  # "hf" only if vLLM cannot start on this GPU (never mix within a model)
)
N_SAMPLES = 4
TEMPERATURE = 0.7  # 0 = greedy G0 run (1 sample per problem, own folder)
SCREEN_PROMPTS = ["round0"]  # ["plain"] for the G0 sanity run
REF = "main"  # branch, tag or commit of the repository to run
REPO_URL = "https://github.com/Arshia-HZ/mas-channel-audit.git"
USE_DRIVE = (
    True  # Colab only: keep results in Google Drive so they survive a disconnect
)

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB:
    REPO = "/content/mas-channel-audit"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/mas-channel-audit"
    else:
        RUN_ROOT = "/content/mas-channel-audit-runs"
elif IN_KAGGLE:
    REPO = "/kaggle/working/mas-channel-audit"
    RUN_ROOT = "/kaggle/working/runs"
else:  # a local checkout
    REPO = os.getcwd()
    if os.path.basename(REPO) == "notebooks":
        REPO = os.path.dirname(REPO)
    RUN_ROOT = os.environ.get("MCA_RUN_ROOT", os.path.join(REPO, "runs"))
os.makedirs(RUN_ROOT, exist_ok=True)


def sh(*cmd, cwd=None, check=True):
    """Run a command, show its output in the cell, raise if it fails."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(proc.stdout, end="")
    print(proc.stderr, end="", file=sys.stderr)
    if check and proc.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {proc.returncode}")
    return proc


print("colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", "| repo:", REPO)
print("results go to:", RUN_ROOT)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    if not os.path.isdir(f"{REPO}/.git"):
        sh("git", "clone", "-q", REPO_URL, REPO)
    sh("git", "fetch", "-q", "--all", cwd=REPO)
    sh("git", "checkout", "-q", REF, cwd=REPO)
    if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO).returncode == 0:
        sh("git", "pull", "-q", "--ff-only", cwd=REPO)  # on a branch: take the latest
    extra = "vllm" if BACKEND == "vllm" else "hf"
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", f".[{extra},data]", cwd=REPO)

os.chdir(REPO)
sys.path.insert(
    0, f"{REPO}/src"
)  # a running kernel does not see a fresh editable install
sh("git", "log", "--oneline", "-1")

**Optional: Hugging Face token.** Only the gated Llama model needs one (store it as a Colab
secret or Kaggle secret named `HF_TOKEN`). The token is never printed.

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token and IN_COLAB:
    from google.colab import userdata

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
if not token and IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # the secret is simply not set
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "not set (fine for the Qwen models)")

In [ ]:
import shutil
from importlib import metadata

if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
for pkg in ("vllm", "torch", "transformers", "numpy", "math-verify", "pydantic"):
    try:
        print(f"{pkg:<14}{metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## 2. Model and problems

In [ ]:
from channel_audit.models import load_model_config

model = load_model_config(f"{REPO}/{MODEL_CONFIG}")
if model.backend != BACKEND:
    model = model.model_copy(update={"backend": BACKEND})
slug = model.model_id.split("/")[-1].lower()
RUN = f"{RUN_ROOT}/screen_{slug}_{BACKEND}"
if TEMPERATURE != 0.7:
    RUN += f"_t{TEMPERATURE:g}".replace(".", "p")
print(model.model_id, "@", model.revision[:12], "|", model.backend, model.dtype)
print("max_tokens:", model.max_tokens, "| max_model_len:", model.max_model_len)
print("run directory:", RUN)

In [ ]:
from channel_audit.data import (
    MATH_REVISION,
    assign_splits,
    load_frozen,
    load_math_test,
    write_split,
)

SPLIT_DIR = os.environ.get("MCA_SPLIT_DIR", f"{REPO}/data/problems")
# Rebuilds the split files from the pinned MATH revision and checks them against the
# committed manifest; refuses to continue if anything differs.
rows, dropped = load_math_test()
manifest = write_split(
    assign_splits(rows), SPLIT_DIR, n_rows=len(rows) + len(dropped), dropped=dropped
)
problems = load_frozen(SPLIT_DIR, "screen")
print("MATH revision:", MATH_REVISION)
print(
    "screen split:",
    len(problems),
    "problems, sha256",
    manifest["files"]["screen"]["sha256"][:16],
)

## 3. Generate

The model is loaded **once** (a minute or two). 300 problems x 4 samples = 1,200 generations,
run 50 problems at a time and saved after each batch.

In [ ]:
from channel_audit.runner import (
    backend_for_run,
    config_for_model,
    create_run,
    grade_pending,
)
from channel_audit.screening import resolve_samples, run_screen

In [ ]:
n_samples = resolve_samples(TEMPERATURE, None if TEMPERATURE == 0 else N_SAMPLES)
cfg = config_for_model(model, replicates=1, log_prompts=True, temperature=TEMPERATURE)
ctx = create_run(RUN, cfg, problems, backend_for_run(cfg, problems))  # loads the model
print(f"{len(problems)} problems; {len(ctx.gens)} generations already stored")
for prompt in SCREEN_PROMPTS:
    run_screen(ctx, n_samples=n_samples, prompt=prompt)
print(f"{len(ctx.gens)} generations stored")

In [ ]:
from channel_audit.runner import throughput_summary

t = throughput_summary(ctx.out_dir)
print(f"{grade_pending(ctx)} generations graded")
print(
    f"throughput: {t['completion_tokens_per_s']:.0f} completion tokens/s "
    f"({t['completion_tokens']} tokens, {t['seconds']:.0f} s, {t['n_batches']} backend calls)"
)

## 4. Report

In [ ]:
import json

from channel_audit.screening import format_comparison, screen_report

reports = [screen_report(ctx, p) for p in SCREEN_PROMPTS]
print(format_comparison(reports))
for prompt, report in zip(SCREEN_PROMPTS, reports, strict=True):
    name = (
        "screen_report.json" if prompt == "round0" else f"screen_report_{prompt}.json"
    )
    with open(f"{RUN}/{name}", "w", encoding="utf-8") as fh:
        json.dump(report, fh, indent=1, sort_keys=True)

Read the table like this:

- **acc / acc3-5**: share of the 1,200 samples that are correct, on all levels and on levels 3 to 5.
- **disagree**: share of problems whose parsable answers are not all equivalent.
- **mixed3-5**: share of level 3 to 5 problems with at least one correct and one incorrect
  sample. This is the headroom a channel could use; the plan picks the primary model by
  usable round-0 disagreement.
- **trunc / extr.fail**: samples cut off at `max_tokens`, and samples that finished without a
  `\boxed{}` answer (extraction failure; G0 needs it under 5%).

## 5. Save the results

In [ ]:
import shutil

archive = shutil.make_archive(f"{RUN}_results", "zip", root_dir=RUN)
print("saved:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(archive)

**Send back** the printed table for each model and its zip. After all four runs, compare them
side by side with `python scripts/04_screen_report.py --runs <run dirs>`.